# LangGraph Single-Agent Setup
## Problem Statement
Wire DevContext AI as a LangGraph state machine with LLM-based intent routing.

## My Approach

Drew the graph on paper first: one entry node for intent classification, then two separate branches. Kept `intent_router` as a plain Python function feeding into `add_conditional_edges` - not a graph node. Split the impact branch into three nodes (`diff_parser → dependency_finder → impact_generate`) so each step could be tested in isolation. All nodes return only the keys they update; state accumulates across the pipeline.

## What I Learned

- `add_conditional_edges` takes a **function**, not a node name - the function reads state and returns a string that maps to a node. Got this wrong the first time by registering `intent_router` as a node.
- LangGraph state is **accumulated, not replaced** - initializing list fields as `""` instead of `[]` causes silent failures downstream because empty string is falsy.
- Splitting the impact pipeline into 3 nodes feels like overkill for a mock, but it's the right call: each node is independently unit-testable, and `dependency_finder` can be swapped for a real ChromaDB call without touching anything else.
- The LLM router pattern works identically on OpenAI or Anthropic - `max_tokens=5` keeps the classification call cheap regardless of provider.

## Where I Got Stuck

Two bugs cost real time:

**Bug 1 - `intent_router` registered as a node.** I added it with `graph.add_node("intent_router", intent_router)` and the graph compiled fine - no error. But the conditional edge was calling the function directly anyway, so the node was just dead weight. Only caught it by printing the ASCII graph and noticing an orphaned node with no incoming edge.

**Bug 2 - list fields initialized as `""`.**  `"sources": ""`, `"changed_functions": ""`, `"impacted_files": ""` all passed TypedDict silently. `dependency_finder` does `if not changed_fns` - empty string is falsy, so it returned immediately with no results every time. Took 20 minutes to trace. Fix was trivial: `[]` instead of `""`.

## What I'd Do Differently

- Initialize state in a helper function or dataclass so there's no chance of `""` vs `[]` mistakes at invocation time.
- Replace the keyword-match retrieval in `qa_retrieve_generate` with the actual ChromaDB index - the node signature doesn't change at all, just the internals.
- Log every routing decision: `{"query": ..., "mode_classified": ..., "timestamp": ...}` - two lines of code that would make misclassification debugging trivial.

In [1]:
MOCK_CODEBASE = {
    "authenticate_user": """
def authenticate_user(user_id: str) -> dict:
    \"\"\"
    Validates user credentials against the users table.
    Fetches user record, checks account_status != 'suspended',
    generates a signed JWT with 24h expiry, logs auth event to audit_log.
    Raises AuthError if user not found or suspended.
    Returns: {"token": str, "expires_at": datetime, "user_id": str}
    \"\"\"
""",
    "get_user_profile": """
def get_user_profile(user_id: str, include_prefs: bool = False) -> dict:
    \"\"\"
    Fetches user metadata from users + user_preferences tables.
    Calls authenticate_user(user_id) internally to validate session.
    If include_prefs=True, joins user_preferences on user_id.
    Returns: {"id", "email", "role", "created_at", "preferences"?}
    \"\"\"
""",
    "create_order": """
def create_order(user_id: str, items: list[dict], payment_method_id: str) -> dict:
    \"\"\"
    Validates cart, reserves inventory, calls authenticate_user(user_id) 
    before processing. Writes to orders + order_items tables in a transaction.
    Emits order.created event to Kafka topic.
    Returns: {"order_id": str, "status": "pending", "estimated_delivery": date}
    \"\"\"
""",
    "refresh_token": """
def refresh_token(expired_token: str) -> dict:
    \"\"\"
    Decodes expired JWT (ignores expiry), extracts user_id,
    calls authenticate_user(user_id) to revalidate account status.
    Issues a new token. Blacklists the old one in Redis.
    Raises TokenError if token is malformed or user is suspended.
    \"\"\"
"""
}

MOCK_DOCS = {
    "auth-flow.md": """
# Authentication Flow

All protected endpoints require a Bearer token in the Authorization header.

## Token lifecycle
- Issued by `authenticate_user()` in `auth.py`
- JWT payload: `{user_id, role, iat, exp}`
- Expiry: 24h. Refresh via `POST /auth/refresh`
- Blacklist stored in Redis (`token:blacklist:<jti>`)

## Session invalidation
Tokens are invalidated on: password change, account suspension, explicit logout.
`refresh_token()` revalidates account status on every refresh - suspended accounts 
cannot renew even with a valid unexpired token.

## Error codes
- 401 Unauthorized - token missing or malformed
- 403 Forbidden - valid token but suspended account
""",
    "db-schema.md": """
# Core Tables

## users
| column         | type        | notes                          |
|----------------|-------------|--------------------------------|
| id             | UUID PK     |                                |
| email          | VARCHAR     | unique                         |
| account_status | ENUM        | active / suspended / deleted   |
| created_at     | TIMESTAMP   |                                |

## orders
| column            | type      | notes                        |
|-------------------|-----------|------------------------------|
| id                | UUID PK   |                              |
| user_id           | UUID FK   | → users.id                   |
| status            | ENUM      | pending/confirmed/shipped    |
| payment_method_id | UUID FK   |                              |

## audit_log
| column     | type      | notes                          |
|------------|-----------|--------------------------------|
| event_type | VARCHAR   | e.g. auth.success, auth.fail   |
| user_id    | UUID FK   |                                |
| created_at | TIMESTAMP |                                |
"""
}

SAMPLE_DIFF = """
--- a/auth.py
+++ b/auth.py
@@ -12,7 +12,7 @@
-def authenticate_user(user_id: str) -> dict:
+def authenticate_user(user_id: str, token: str = None) -> dict:
     \"\"\"
-    Validates user credentials. Generates JWT on success.
+    Validates user credentials. If token provided, validates existing session instead of issuing new one.
     \"\"\"
-    user = db.query(f"SELECT * FROM users WHERE id = '{user_id}'")
+    if token:
+        return validate_existing_session(user_id, token)
+    user = db.query("SELECT * FROM users WHERE id = %s", [user_id])
"""

In [2]:
from typing import TypedDict
from langgraph.graph import StateGraph, START , END
from langchain_groq import ChatGroq
import re
from langgraph.errors import GraphRecursionError
llm = ChatGroq(model="llama-3.3-70b-versatile", temperature=0)


d:\Python\genai-engineering-portfolio\venv\Lib\site-packages\langgraph\cache\base\__init__.py:8: LangChainPendingDeprecationWarning: The default value of `allowed_objects` will change in a future version. Pass an explicit value (e.g., allowed_objects='messages' or allowed_objects='core') to suppress this warning.
  from langgraph.checkpoint.serde.jsonplus import JsonPlusSerializer
d:\Python\genai-engineering-portfolio\venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [3]:

class AgentState(TypedDict):
    query: str              # "user query"
    mode: str               # "qa" or "impact"
    retrieved_context: str  # "docs/code chunks retrieved from ChromaDB"
    answer: str             # llm answer
    sources: list[str]      # "Sources retrieved from chromadb"
    changed_functions: list[str] # extracted from PR diff
    impacted_files: list[str]   # files that call changed functions

In [4]:
#entry node
def route_intent(state: AgentState) -> dict :
    prompt = f"""
    Classify below query into qa or pr_impact .
    qa=asking about how code/system works or onboarding questions
    pr_impact=asking about what code breaks if code changes are done , pr review questions.
    Query = {state['query']}
    Respond with one word only : qa or pr_impact.
    """
    response = llm.invoke(prompt , max_tokens=5)
    mode=response.content.strip().lower()
    if mode not in ('qa','pr_impact'):
        mode='qa' #fallback
    return {"mode":mode}

#Mapping intent 
def intent_router(state: AgentState) -> str:
    return state["mode"]  # "qa" -> qa branch, "pr_impact" ->  pr branch 

#QA answer generate , retrieval would be from chromadb and it would have been separate function in production case.
def qa_retrieve_generate(state: AgentState) -> dict:
    query_lower = state["query"].lower()
    relevant_code = {
        k: v for k, v in MOCK_CODEBASE.items()
        if any(word in query_lower for word in k.split("_")) or k in query_lower
    }
    if not relevant_code:
        first_key = next(iter(MOCK_CODEBASE))
        relevant_code = {first_key: MOCK_CODEBASE[first_key]}

    relevant_docs = {
        k: v for k, v in MOCK_DOCS.items()
        if any(word in v.lower() for word in query_lower.split())
    }

    context_code_str = "\n".join(f"[codebase:{k}]\n{v}" for k, v in relevant_code.items())
    context_doc_str  = "\n".join(f"[docs:{k}]\n{v}"     for k, v in relevant_docs.items())

    prompt = (
        "You are a codebase assistant. Answer using ONLY the context below.\n"
        "Cite source keys used (e.g. codebase:authenticate_user, docs:auth-flow.md).\n\n"
        f"--- Codebase ---\n{context_code_str}\n\n"
        f"--- Docs ---\n{context_doc_str}\n\n"
        f"Query: {state['query']}\n\n"
        "Output format (two lines only):\n"
        "Sources: comma-separated source keys\n"
        "Answer: your answer"
    )
    response = llm.invoke(prompt)
    content = response.content.strip()

    sources, answer = [], content
    for line in content.splitlines():
        if line.lower().startswith("sources:"):
            sources = [s.strip() for s in line.split(":", 1)[1].split(",")]
        elif line.lower().startswith("answer:"):
            answer = line.split(":", 1)[1].strip()

    return {
        "retrieved_context": context_code_str + "\n" + context_doc_str,
        "answer": answer,
        "sources": sources,
    }

#pr impact code.
def diff_parser (state : AgentState) -> dict:
    pattern = r'^[+-][\t ]*def\s+(\w+)\s*\('
    matches=re.findall(pattern , state['query'] , re.MULTILINE)
    return {"changed_functions" :list(set(matches))}

def dependency_finder(state: AgentState ) -> dict :
    changed_fns = state.get("changed_functions", [])
    if not changed_fns:
        return {"retrieved_context": "No changes detected", "impacted_files": []}

    calling_keys = []
    context_chunks = []

    # Iterate through each changed function found by the diff_parser
    for fn_name in changed_fns:
        clean_fn = fn_name.strip()
        call_pattern = re.compile(fr'\b{re.escape(clean_fn)}\s*\(') 
        
        for file_name, code_content in MOCK_CODEBASE.items():
            if call_pattern.search(code_content):
                calling_keys.append(file_name)
                context_chunks.append(f"--- {file_name} ---\n{code_content}")

    return {
        "retrieved_context": "\n".join(list(set(context_chunks))), 
        "impacted_files": list(set(calling_keys))
    }

def impact_generate(state:AgentState) -> dict :
    prompt = f"""You are a code impact analyst.
    On basis of context and impacted files , Generate an answer grounded from it .
    Context = {state["retrieved_context"]}
    Impacted files = {state["impacted_files"]}

    """
    response = llm.invoke(prompt)
    return {"answer": response.content}

    

In [5]:
graph = StateGraph(AgentState)
graph.add_node('route_intent' , route_intent)
# graph.add_node("intent_router", intent_router)
graph.add_node('qa_retrieve_generate' , qa_retrieve_generate)
graph.add_node('diff_parser' , diff_parser)
graph.add_node('dependency_finder' , dependency_finder)
graph.add_node('impact_generate' , impact_generate)

graph.add_edge(START, 'route_intent')
graph.add_conditional_edges('route_intent' , intent_router , 
{
        "qa": "qa_retrieve_generate", 
        "pr_impact": "diff_parser"
})

graph.add_edge('qa_retrieve_generate' , END)
graph.add_edge('diff_parser' , 'dependency_finder')
graph.add_edge('dependency_finder' , 'impact_generate')
graph.add_edge('impact_generate' , END)

app=graph.compile()


In [6]:
# Visualize
print(app.get_graph().draw_ascii())

                    +-----------+                        
                    | __start__ |                        
                    +-----------+                        
                           *                             
                           *                             
                           *                             
                   +--------------+                      
                   | route_intent |                      
                   +--------------+                      
                  ..              ...                    
               ...                   ..                  
             ..                        ...               
   +-------------+                        ..             
   | diff_parser |                         .             
   +-------------+                         .             
          *                                .             
          *                                .             
          *   

In [7]:
qa_result = app.invoke({
    "query": "What does the `authenticate_user` function do?",
    "mode": "",
    "retrieved_context": "",
    "answer": "",
    "sources":[],
    "changed_functions":[],
    "impacted_files":[]
})
print(f"Mode detected: {qa_result['mode']}")
print(f"retrieved_context: {qa_result['retrieved_context']}")
print(f"sources: {qa_result['sources']}")
print(f"Answer: {qa_result['answer']}")

Mode detected: qa
retrieved_context: [codebase:authenticate_user]

def authenticate_user(user_id: str) -> dict:
    """
    Validates user credentials against the users table.
    Fetches user record, checks account_status != 'suspended',
    generates a signed JWT with 24h expiry, logs auth event to audit_log.
    Raises AuthError if user not found or suspended.
    Returns: {"token": str, "expires_at": datetime, "user_id": str}
    """

[codebase:get_user_profile]

def get_user_profile(user_id: str, include_prefs: bool = False) -> dict:
    """
    Fetches user metadata from users + user_preferences tables.
    Calls authenticate_user(user_id) internally to validate session.
    If include_prefs=True, joins user_preferences on user_id.
    Returns: {"id", "email", "role", "created_at", "preferences"?}
    """

[docs:auth-flow.md]

# Authentication Flow

All protected endpoints require a Bearer token in the Authorization header.

## Token lifecycle
- Issued by `authenticate_user()` in

In [32]:
qa_result = app.invoke({
    "query": f"""PR diff: 
-def authenticate_user(user_id) 
+def authenticate_user(user_id, token)""",
    "mode": "",
    "retrieved_context": "",
    "answer": "",
    "sources":[],
    "changed_functions":[],
    "impacted_files":[]
})
print(f"Mode detected: {qa_result['mode']}")
print(f"retrieved_context: {qa_result['retrieved_context']}")
print(f"sources: {qa_result['sources']}")
print(f"Answer: {qa_result['answer']}")

Mode detected: pr_impact
retrieved_context: --- authenticate_user ---

def authenticate_user(user_id: str) -> dict:
    """
    Validates user credentials against the users table.
    Fetches user record, checks account_status != 'suspended',
    generates a signed JWT with 24h expiry, logs auth event to audit_log.
    Raises AuthError if user not found or suspended.
    Returns: {"token": str, "expires_at": datetime, "user_id": str}
    """

--- refresh_token ---

def refresh_token(expired_token: str) -> dict:
    """
    Decodes expired JWT (ignores expiry), extracts user_id,
    calls authenticate_user(user_id) to revalidate account status.
    Issues a new token. Blacklists the old one in Redis.
    Raises TokenError if token is malformed or user is suspended.
    """

--- get_user_profile ---

def get_user_profile(user_id: str, include_prefs: bool = False) -> dict:
    """
    Fetches user metadata from users + user_preferences tables.
    Calls authenticate_user(user_id) internal